Some info about the data:
- the neural data is recorded at 30000 Hz
- the behavioral data is recorded at 40 Hz (we refer to the behavioral sampling times as frames)
- threatening stimuli are usually an air puff delivered simultaneously with an auditory threat (sometimes it's only an auditory threat). The threats are always delivered in the same place at exactly the opposite end of the arena compared to the shelter.
- The video data is collected at 1024x1024 pixels (~10pixels per cm). All positional information is in pixels.

# Setup
## Imports

In [1]:
import numpy as np
import os
import polars as pl
import dill as pickle
import socket
import tkinter as tk
from tkinter import filedialog

import matplotlib
import matplotlib.pyplot as plt

## Select Ceph mount

In [ ]:
"""The paths to all the sessions"""
# base path: identifies where you have mapped ceph onto your computer
# all_paths: a list of the paths to specific experimental sessions 
# NB: some paths are commented out! This is because we are migrating some of our data between servers right now. It should be available on ceph in a few days.
# Also note: JAL<00X> is the naming convention for the different animals

def get_computer_specific_paths():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    base_path = filedialog.askdirectory(title="Select Directory in which you have mounted Branco lab ceph")
    return base_path

base_path = get_computer_specific_paths()

all_paths = ['Jasmine_Laurence/Experimental_Data/JAL004/JAL004_flip_rotated_2023_08_28T09_36_04',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flip_2023_09_03T12_04_16',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flip_puff2_2023_09_11T09_32_25',
             'Jasmine_Laurence/Experimental_Data/JAL004/004_flipppuf19sept_2023_09_19T14_10_56',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_baseline_2023_09_05T07_48_58',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_flip1_2023_09_08T07_36_54',
             'Jasmine_Laurence/Experimental_Data/JAL005/005_flippuff3_2023_09_21T11_11_13',
            #  'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_barrier_flip3_2024_03_18T11_53_29',
            #  'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34',
            #  'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33',
            #  'Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_6_2024_03_28T10_54_20',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_barrierflip2_2024_03_12T11_18_26',
             'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_5_2024_03_22T11_15_43',
            #  'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_8_2024_04_09T10_07_45',
            #  'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_9_2024_04_16T11_13_05',
            #  'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_shelter_barrier_flip_100_2024_04_23T09_59_40',
            #  'Jasmine_Laurence/Experimental_Data/JAL007/JAL007_tinnybarrier1_2024_04_30T10_57_04',
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_1_2024_04_25T11_27_42',
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_2_2024_04_29T12_14_54',,
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_tiny_barrier_flip_1_2024_05_03T10_02_35'
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26',
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_4_2024_05_10T11_47_47',
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_barrier_flip_5_2024_05_14T10_18_03',
            #  'Jasmine_Laurence/Experimental_Data/JAL008/JAL008_shelter_tiny_flip_2_2024_05_21T11_10_19'
             ]
print(base_path)

: 

## Load behavioural data

In [ ]:
# Loading in even one session takes a fair while (and a lot of memory!)
# the code below is responsive to multiple sessions, but use this if necessary to restrict to a subset of sessions

all_paths = [all_paths[4], all_paths[5]]

In [ ]:
# These are the columns of the dataframe
# -- frames: the behavioral frame number (this will match the rows of the time x neurons matrix and the frames in spike_df['spike_aligned_to_frame'])
# -- hdir: the mouse's head direction 
# -- hsa: the mouse's head-shelter angle
# -- mouse_x_position
# -- mouse_y_position
# -- OutofshelterIdx (bool): If True the mouse was outside the shelter
# -- EscapePeriod (bool): If True the mouse is performing an escape
# -- shelter (bool): If True the shelter is present in the arena
# -- barrier_present (bool): If True the barrier is present in the arena (to only look at times when there is no barrier you can use this to fin the frames before this becomes True)
# -- barrier_flipped (bool): If True the barrier has been flipped 180deg
# -- speed
# -- homingPeriod (bool): If True the mouse is performing a homing run
# -- h_preflipbar_a: IGNORE: the angle between the mouse's head and the open side of the barrier before flipping it
# -- h_postflipbar_a: IGNORE: the angle between the mouse's head and the open side of the barrier after flipping it
# -- h_bar_centre_a: IGNORE: the angle between the mouse's head and the centre of the barrier (also the centre of the arena)

# Function for computing an additional angular velocity column (difference in hd between successive frames)
def get_av(theta_in: pl.Series) -> pl.Series:
    theta_ = theta_in.to_numpy().copy()

    for i in range(1, len(theta_)):
        d_theta = theta_[i] - theta_[i - 1]
        if d_theta > np.pi:
            theta_[i:] -= 2 * np.pi
        elif d_theta < -np.pi:
            theta_[i:] += 2 * np.pi

    theta_dot = np.concatenate(([np.nan], np.diff(theta_)))

    return pl.Series(name="av", values=theta_dot)

# Scan all sessions and save only frames within relevant period (out of shelter, no escape, shelter present, no barrier, no homing)
all_behave_df = None
for exp_idx, exp_path in enumerate(all_paths):
    session_df = (
        pl.scan_csv(os.path.join(base_path, exp_path, "processed_data", "full_video_dataframe.csv"))
        .filter(
            (pl.col("OutofshelterIdx") == True) &
            (pl.col("EscapePeriod") == False) &
            (pl.col("shelter") == True) &
            (pl.col("barrier_present") == False) &
            (pl.col("homingPeriod") == False)
        )
        .select(['frames', 'hdir', 'mouse_x_position', 'mouse_y_position', 'speed', 'hsa'])
        .collect()
    )

    theta = session_df['hdir']
    av = get_av(theta)

    session_df = session_df.with_columns(
        (pl.lit(exp_idx).alias("exp_idx"),
         av)
    )

    print('Loaded session', exp_path)
    print(session_df)
    print()

    if all_behave_df is None:
        all_behave_df = session_df
    else:
        all_behave_df = pl.concat([all_behave_df, session_df], how='vertical')
    del session_df

all_behave_df.head()

## Load spikes dataframe

In [ ]:
# These are the columns of the dataframe
# -- ['aligned_spike_times']: IGNORE (spike times in behavioral computer clock)
# -- ['spike_aligned_to_frame']: the frame (behavioral sampling timepoint) that a given spike was recorded on
# -- ['aligned_spike_times_in_samples']: IGNORE (spike times in behavioral computer clock)
# -- ['spike_times']: IGNORE  (spike time in recording computer clock)
# -- ['spike_clusters']: the cluster that a given spike belongs to
# -- ['cluster_group']: the spikesorting classification of a given cluster as 'good' (putative single unit), 'mua' (multiunit activity), 'noise' (noise cluster)

# Scan all sessions and save only spikes in frames of relevant behaviour
all_spikes_df = None
for exp_idx, exp_path in enumerate(all_paths):
    session_timesteps = all_behave_df.filter(pl.col("exp_idx") == exp_idx).unique('frames').select('frames').sort('frames').to_series()
    session_df = (
        pl.scan_csv(os.path.join(base_path, exp_path, "processed_data", "Processed_efizz_data"))
        .filter(
            (pl.col("cluster_group") == "good") &
            (pl.col("spike_aligned_to_frame").is_in(session_timesteps))
        )
        .select(['spike_aligned_to_frame', 'spike_clusters'])
        .rename({"spike_aligned_to_frame": "t", "spike_clusters": "i"})
        .collect()
    )

    session_df = session_df.with_columns(
        (pl.lit(exp_idx).alias("exp_idx"),
         pl.col("t").cast(pl.Int64))
    )

    print('Loaded session', exp_path)
    print(session_df)
    print()

    if all_spikes_df is None:
        all_spikes_df = session_df
    else:
        all_spikes_df = pl.concat([all_spikes_df, session_df], how='vertical')
    del session_df

all_spikes_df = all_spikes_df.with_columns(
    (pl.col('exp_idx')*1e5 + pl.col('i')).alias('i').cast(pl.Int64)
)

all_spikes_df.head()

## Other data

In [ ]:
# """Get data paths and session metadata for a given session"""
# session object has a lot of information about the recording. The following could be useful to you:
# session.audio.onset_frames is a list (sometimes it's a list of lists sorry!) of times in behavioral frames of when the threat was delivered
# session.shelter_location gives you the xy position of the top left and bottom right corners of the shelter
# session.barrier_time gives you the time in minutes when the barrier was first placed in the arena (you probably want everything before this time)

# with open(os.path.join(base_path, experiment_path, "processed_data", "metadata"), "rb") as dill_file: 
#     session = pickle.load(dill_file)

# """Produces ModuleNotFound error: no module behave_analysis"""

In [ ]:
# """"Load time x neurons matrix"""
# this will give you the firing rate of each putative single unit at each frame (behavioral sampling timepoint)
# frame_by_cluster_matrix = np.load(os.path.join(base_path, experiment_path, "processed_data") + "/" + "frame_by_good_cluster_matrix.npy")

# Pre-Processing

## Behavioural

In [ ]:
# clean up dataframe for use in tuning computations

behave_df = (
    all_behave_df

    .with_columns(
        (pl.col("hdir") % (2 * np.pi)).alias("hd"),
        (pl.col("hsa") % (2 * np.pi)).alias("sd")
    )
    .drop(["hdir", "hsa"])

    .with_columns(
        (pl.col("mouse_x_position")).alias("x"),
        (pl.col("mouse_y_position")).alias("y")
    ).
    drop(["mouse_x_position", "mouse_y_position"])

    .with_columns(
        (pl.col("speed")).alias("v")
    ).drop("speed")

    .rename({
        "frames": "t"
    })
)

# behave_df = select_behave_df.filter(pl.col('v') < 1)

behave_df.head()

## Spikes

In [ ]:
valid_frames = behave_df.unique(['exp_idx', 't']).select(['exp_idx', 't'])
valid_clusters = all_spikes_df.unique(['exp_idx', 'i']).select(['exp_idx', 'i'])

spike_counts = (
    all_spikes_df
    .group_by(["exp_idx", "t", "i"])
    .agg(pl.len().alias("spikes_per_frame"))
)

# Create all (session, frame, cluster) combinations, fill missing counts with 0
all_combinations = (
    valid_frames
    .join(valid_clusters, on='exp_idx', how='right')
)

spike_counts = (
    all_combinations
    .join(spike_counts, on=["exp_idx", "t", "i"], how="left")
    .fill_null(0)
)


stats = (
    spike_counts
    .group_by("i")
    .agg([
        pl.col("spikes_per_frame").mean().alias("mean_r_overall"),
        pl.col("spikes_per_frame").std().alias("std_r_overall"),
    ])
)

# -------------
# 3) Rolling window and z-score calculation
# -------------
fps = 40
frame_time = 1 / fps
window_time = 0.1
window_size = int(window_time // frame_time) 


spikes_df = (
    spike_counts
    .join(stats, on=["i"], how="left")
    .sort(["i", "t"])
    .with_columns(
        # Rolling mean over the last 'window_size' frames
        pl.col("spikes_per_frame").rolling_mean(window_size).over("i").alias("r")
    )
    # Drop rows that cannot have a full window
    .drop_nulls("r")
    # c is just spikes_per_frame at the start of the window (the current row)
    .with_columns(pl.col("spikes_per_frame").alias("c"))
    # We'll track the frame index at the start of the window
    .with_columns(pl.col("t").alias("t_start_window"))
    # Compute z = (r - mean_overall) / std_overall (or zero if std=0)
    .with_columns(
        pl.when(pl.col("std_r_overall") == 0)
        .then(0)
        .otherwise((pl.col("r") - pl.col("mean_r_overall")) / pl.col("std_r_overall"))
        .alias("z")
    )
    .select(["exp_idx", "t_start_window", "i", "z", "r", "c"])
    .rename({"t_start_window": "t"})
)
del spike_counts, all_combinations, stats

spikes_df



In [ ]:
N = 2  # set the number of rows you want
top_exps = (
    behave_df
    .group_by("exp_idx")
    .agg(pl.count("t").alias("n_timeframes"))
    .sort("n_timeframes", descending=True)
    .head(N)
    .select("exp_idx").to_series().to_list()[-1]
)
print(top_exps)

behave_df.filter(pl.col("exp_idx").is_in(top_exps)).write_csv("top_behave_df.csv")
spikes_df.filter(pl.col("exp_idx").is_in(top_exps)).write_csv("top_spikes_df.csv")

In [ ]:
unique_exps = behave_df.select('exp_idx').unique().to_series().sort().to_list()
plt.figure(figsize=(8, 6))
for e in unique_exps:
    subset = behave_df.filter(pl.col('exp_idx') == e)
    angle_sum = ((subset['hd'] + subset['sd']) % (2 * np.pi)).to_numpy()
    plt.hist(angle_sum, bins=60, alpha=0.5, label=f'exp_idx {e}')
plt.xlabel('Angle (radians)')
plt.ylabel('Frequency')
plt.legend()
plt.title('Histogram of (hd + sd) mod 2π by exp_idx')
plt.show()

In [ ]:
neurons = spikes_df.unique('i').sort(['exp_idx', 'i']).select('i').to_series().to_list()

bounding_timesteps = {}
bounding_neurons = {}
for exp_idx in behave_df.unique('exp_idx').sort('exp_idx').select('exp_idx').to_numpy().flatten():
    exp_behave_df = behave_df.filter(pl.col('exp_idx')==exp_idx)
    old_first_timestep = exp_behave_df.select('t').min().to_numpy().flatten()[0]
    old_last_timestep = exp_behave_df.select('t').max().to_numpy().flatten()[0]

    timestep_offset = -old_first_timestep if exp_idx==0 else bounding_timesteps[exp_idx-1][1] - old_first_timestep
    behave_df = behave_df.with_columns((pl.when(pl.col('exp_idx')==exp_idx).then(pl.col('t') + timestep_offset).otherwise(pl.col('t'))).alias('t'))
    spikes_df = spikes_df.with_columns((pl.when(pl.col('exp_idx')==exp_idx).then(pl.col('t') + timestep_offset).otherwise(pl.col('t'))).alias('t'))

    new_first_timestep = old_first_timestep + timestep_offset
    new_old_timestep = old_last_timestep + timestep_offset
    bounding_timesteps[exp_idx] = (new_first_timestep, new_old_timestep+1)

    exp_spikes_df = spikes_df.filter(pl.col('exp_idx')==exp_idx)
    first_neuron = exp_spikes_df.select('i').min().to_numpy().flatten()[0]
    first_neuron_i = neurons.index(first_neuron)
    last_neuron = exp_spikes_df.select('i').max().to_numpy().flatten()[0]
    last_neuron_i = neurons.index(last_neuron)
    bounding_neurons[exp_idx] = (first_neuron_i, last_neuron_i)

In [ ]:
spikes_df

# Tuning

In [ ]:
from typing import List, Dict, Tuple
def get_tuning_generalised(tuning_vars_list: List[str], use: str='z', **kwargs) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    global spikes_df, behave_df

    activity_var = use
    assert activity_var in spikes_df.columns, f"{activity_var} is not a column in the dataframe"
    assert activity_var == 'z' or activity_var == 'r', "activity_var should be 'z' or 'r'"
    assert len(tuning_vars_list) > 0, "tuning_vars_list should not be empty"

    n_angle_bins = kwargs.get('n_angle_bins', 360)
    n_position_bins = kwargs.get('n_position_bins', 100)
    n_V_bins = kwargs.get('n_V_bins', 100)
    n_V_std = kwargs.get('n_V_std', 3)

    neurons = spikes_df.unique('i').sort(['exp_idx', 'i']).select('i').to_series().to_list()
    n_neurons = len(neurons)
    
    timesteps = behave_df.unique('t').sort('t').select('t').to_series().to_list()
    n_timesteps = len(timesteps)

    tuning_vars = {}

    print('Computing tuning vars')

    if 'av' in tuning_vars_list:

        AV = behave_df.unique('t').select('t', 'av').sort(by='t').select('av').to_numpy() * 180/np.pi

        std_AV = np.nanstd(AV)
        min_AV = np.nanmin(AV)
        max_AV = np.nanmax(AV)
        min_AV_included = max(-n_V_std * std_AV, min_AV)
        max_AV_included = min(n_V_std * std_AV, max_AV)

        AV_bins = np.linspace(min_AV_included, max_AV_included, n_V_bins+1)[:-1]

        tuning_vars['av'] = dict(var=AV, bins=AV_bins, title='Angular Velocity')

    if 'hd' in tuning_vars_list:

        HD = behave_df.unique('t').select('t', 'hd').sort(by='t').select('hd').to_numpy() * 180/np.pi

        angle_bins = np.linspace(0, 360, n_angle_bins+1)[:-1]

        tuning_vars['hd'] = dict(var=HD, bins=angle_bins, title='Head Direction')

    if 'sd' in tuning_vars_list:
        
        SD = behave_df.unique('t').select('t', 'sd').sort(by='t').select('sd').to_numpy() * 180/np.pi

        angle_bins = np.linspace(0, 360, n_angle_bins+1)[:-1]

        tuning_vars['sd'] = dict(var=SD, bins=angle_bins, title='Head-Shelter Angle')

    if 'x' in tuning_vars_list:

        X = behave_df.unique('t').select('t', 'x').sort(by='t').select('x').to_numpy()

        position_bins = np.linspace(0, 1, n_position_bins)

        tuning_vars['x'] = dict(var=X, bins=position_bins, title='X Position')

    if 'y' in tuning_vars_list:

        Y = behave_df.unique('t').select('t', 'y').sort(by='t').select('y').to_numpy()

        position_bins = np.linspace(0, 1, n_position_bins)

        tuning_vars['y'] = dict(var=Y, bins=position_bins, title='Y Position')

    if 'v' in tuning_vars_list:

        V = behave_df.unique('t').select('t', 'v').sort(by='t').select('v').to_numpy()

        std_V = np.std(V)
        min_V = np.min(V)
        max_V = np.max(V)
        min_V_included = max(-n_V_std * std_V, min_V)
        max_V_included = min(n_V_std * std_V, max_V)

        V_bins = np.linspace(min_V_included, max_V_included, n_V_bins+1)[:-1]

        tuning_vars['v'] = dict(var=V, bins=V_bins, title='Linear Velocity')


    tuning_dict = {}

    activity = spikes_df.select(['i', 't', activity_var]).sort(by=['i', 't']).pivot(index='i', on='t', values=activity_var).drop('i').to_numpy()


    # Create bins
    for i, i_key in enumerate(tuning_vars.keys()):
        for j, j_key in enumerate((list(tuning_vars.keys())[i:])):
            i_var, i_bins = tuning_vars[i_key]['var'], tuning_vars[i_key]['bins']
            j_var, j_bins = tuning_vars[j_key]['var'], tuning_vars[j_key]['bins']

            # Var tuning
            if j == 0:
                bins = np.zeros((n_neurons, len(i_bins)))
                bin_size = np.zeros((n_neurons, len(i_bins)))

                tuning_dict[f'{i_key}_tuning_bins'] = bins
                tuning_dict[f'{i_key}_tuning_bin_size'] = bin_size

            # Var-to-var tuning
            else:
                bins = np.zeros((n_neurons, len(i_bins), len(j_bins)))
                bin_size = np.zeros((n_neurons, len(i_bins), len(j_bins)))

                tuning_dict[f'{i_key}_to_{j_key}_tuning_bins'] = bins
                tuning_dict[f'{i_key}_to_{j_key}_tuning_bin_size'] = bin_size

    print('Computing tuning curves')

    # Fill bins
    for i, i_key in enumerate(tuning_vars.keys()):
        for j, j_key in enumerate((list(tuning_vars.keys())[i:])):
            i_var, i_bins = tuning_vars[i_key]['var'], tuning_vars[i_key]['bins']
            i_bin_indices = (np.digitize(i_var, i_bins)-1).squeeze() # (n_timesteps,)

            j_var, j_bins = tuning_vars[j_key]['var'], tuning_vars[j_key]['bins']
            j_bin_indices = (np.digitize(j_var, j_bins)-1).squeeze() # (n_timesteps,)

            # Var tuning
            if j == 0:
                bins = tuning_dict[f'{i_key}_tuning_bins']
                bin_size = tuning_dict[f'{i_key}_tuning_bin_size']

                for exp_idx, (first_timestep, last_timestep) in bounding_timesteps.items():
                    first_neuron, last_neuron = bounding_neurons[exp_idx]

                    for t in range(first_timestep, last_timestep):

                        bins[first_neuron:last_neuron, i_bin_indices[t]] += activity[first_neuron:last_neuron, t]
                        bin_size[first_neuron:last_neuron, i_bin_indices[t]] += 1

            # Var-to-var tuning
            else:
                bins = tuning_dict[f'{i_key}_to_{j_key}_tuning_bins']
                bin_size = tuning_dict[f'{i_key}_to_{j_key}_tuning_bin_size']

                tuning_key = f'{i_key}_to_{j_key}_tuning'
                tuning_dict[tuning_key] = np.zeros_like(bins)

                for exp_idx, (first_timestep, last_timestep) in bounding_timesteps.items():
                    first_neuron, last_neuron = bounding_neurons[exp_idx]

                    for t in range(first_timestep, last_timestep):

                        bins[first_neuron:last_neuron, i_bin_indices[t], j_bin_indices[t]] += activity[first_neuron:last_neuron, t]
                        bin_size[first_neuron:last_neuron, i_bin_indices[t], j_bin_indices[t]] += 1

            print(f'\tCompleted {i_key}' + (f' and {j_key}' if j>0 else ''))

    # Average bins
    for i, i_key in enumerate(tuning_vars.keys()):
        for j, j_key in enumerate((list(tuning_vars.keys())[i:])):

            # Var tuning
            if j == 0:
                bins = tuning_dict[f'{i_key}_tuning_bins']
                bin_size = tuning_dict[f'{i_key}_tuning_bin_size']

                tuning_key = f'{i_key}_tuning'

            # Var-to-var tuning
            else:
                bins = tuning_dict[f'{i_key}_to_{j_key}_tuning_bins']
                bin_size = tuning_dict[f'{i_key}_to_{j_key}_tuning_bin_size']

                tuning_key = f'{i_key}_to_{j_key}_tuning'
            
            tuning_dict[tuning_key] = np.divide(bins, bin_size, out=np.zeros_like(bins), where=bin_size!=0)

    return tuning_vars, tuning_dict


r_tuning_vars, r_tuning_dict = get_tuning_generalised(['hd', 'sd', 'x', 'y', 'v', 'av'], use='r', n_angle_bins=60, n_position_bins=10, n_V_bins=10)
z_tuning_vars, z_tuning_dict = get_tuning_generalised(['hd', 'sd', 'x', 'y', 'v', 'av'], use='z', n_angle_bins=60, n_position_bins=10, n_V_bins=10)

In [ ]:
tuning_vars, tuning_dict = {}, {}

for key in r_tuning_vars.keys():
    tuning_vars[f'r-{key}'] = r_tuning_vars[key]
for key in z_tuning_vars.keys():
    tuning_vars[f'z-{key}'] = z_tuning_vars[key]

for key in r_tuning_dict.keys():
    tuning_dict[f'r-{key}'] = r_tuning_dict[key]
for key in z_tuning_dict.keys():
    tuning_dict[f'z-{key}'] = z_tuning_dict[key]

## Tuning Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from typing import Callable, List

def neuron_by_neuron_plot(n_neurons: int, plot_closure: Callable[[matplotlib.axes.Axes], int], x_label: str = None, y_label: str = None, legend_closure: Callable[[matplotlib.figure.Figure], matplotlib.axes.Axes] = None, ordering: List[int] = None, **kwargs) -> matplotlib.figure.Figure:

    # Get relevant config parameters

    width = kwargs.get('width', 25)
    height = kwargs.get('height', 25)
    margin = kwargs.get('margin', 0.05)

    # Create fig with config.n_neurons subplots, with a square arrangement
    n_rows = int(np.ceil(np.sqrt(n_neurons)))
    fig, ax = plt.subplots(nrows=n_rows, ncols=n_rows, figsize=(width, height), sharex=True, sharey=True)

    # Generate ordering if not supplied
    if ordering is None:
        ordering = np.arange(n_neurons, dtype=np.int32)

    # Plot on each subplot, going left-to-right, top-to-bottom
    for i in range(n_rows):
        for j in range(n_rows):
            
            # As there may be more subplots than neurons, turn off unused subplots
            if i*n_rows + j >= n_neurons:
                ax[i,j].set_axis_off()
                continue

            # Get the neuron index of this subplot (as defined by ordering)
            neuron = ordering[i*n_rows + j]

            # Plot on the subplot
            plot_closure(ax[i,j], neuron)

            ax[i,j].annotate(
                f'{neuron}',
                xy=(0, 1), xycoords='axes fraction',
                xytext=(+0.5, -0.5), textcoords='offset fontsize',
                fontsize='medium', verticalalignment='top', fontfamily='serif',
                bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0))

    # Aesthetic settings
    if x_label is not None:
        fig.text(0.5, margin/4, x_label, ha='center', fontsize=18)
    if y_label is not None:
        fig.text(margin/4, 0.5, y_label, va='center', rotation='vertical', fontsize=18)

    if legend_closure is not None:
        legend_closure(fig, ax)
    else:
        handles, labels = ax[0,0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower right', ncol=len(handles), markerscale=10)

    plt.subplots_adjust(left=margin, right=1-margin, top=1-margin, bottom=margin)

    return fig




def univar_tuning_plot(plot_vars: List[Tuple[str, str]], tuning_vars: List[str], tuning_dict: Dict[str, np.ndarray], ordering: List[int] = None, og_ordering: List[int] = None, bin_mask: np.ndarray = None, **kwargs) -> matplotlib.figure.Figure:

    n_neurons = tuning_dict[f'{plot_vars[0][0]}_tuning'].shape[0]

    labels = kwargs.get('labels', [tuning_vars[var]['title'] for var, _ in plot_vars])
    title = kwargs.get('title', None)

    if ordering is None:
        ordering = [i for i in range(n_neurons)]
    else:
        if max(ordering) >= n_neurons:
            assert og_ordering is not None, "Original ordering must be provided if ordering is out of bounds"

    def _plot_angle_tuning(ax, neuron):
        neuron_i = og_ordering.index(neuron)
        for i, (var, col) in enumerate(plot_vars):
            bins = tuning_vars[var]['bins']
            tuning = tuning_dict[f'{var}_tuning']
            bin_counts = tuning_dict[f'{var}_tuning_bin_size']

            plot_tuning = tuning[neuron_i]
            clt_mask = bin_counts[neuron_i] < 30
            plot_tuning[clt_mask] = np.nan
            if bin_mask is not None:
                bins = bins[bin_mask]
                plot_tuning = plot_tuning[bin_mask]

            ax.plot(bins, plot_tuning, color=col, label=labels[i], zorder=0)

        ax.set_ylim([0,1])

    # Use neuron_by_neuron_plot template to create plot
    fig = neuron_by_neuron_plot(n_neurons,
                                 plot_closure=_plot_angle_tuning,
                                 x_label='Variable',
                                 y_label='Activity', 
                                 ordering=ordering, **kwargs)

    if title is not None:
        fig.suptitle(title, fontsize=20)

    return fig

In [ ]:
# ordering = mi_hd_df.sort('mi_hd', descending=True).select('i').to_series().to_list()
# og_ordering = mi_hd_df.sort('i').select('i').to_series().to_list()

ordering = og_ordering = spikes_df.unique('i').sort('i').select('i').to_series().to_list()

fig = univar_tuning_plot([('r-hd', 'red'), ('z-hd', 'orange')], tuning_vars, tuning_dict, ordering=ordering, og_ordering=og_ordering, labels=['Firing rate (spikes/100ms)', 'Z-Scored Firing Rate'], title='Head Direction Tuning - JAL005 Baseline')

In [ ]:
# ordering = mi_sd_df.sort('mi_sd', descending=True).select('i').to_series().to_list()
# og_ordering = mi_sd_df.sort('i').select('i').to_series().to_list()

ordering = og_ordering = spikes_df.unique('i').sort('i').select('i').to_series().to_list()

fig = univar_tuning_plot([('r-sd', 'red'), ('z-sd', 'orange')], tuning_vars, tuning_dict, ordering=ordering, og_ordering=og_ordering, labels=['Firing Rate (spikes/100ms)', 'Z-Scored Firing Rate'], title='Head-Shelter Angle Tuning - JAL005 Baseline')

In [ ]:
def bivar_tuning_plot(plot_vars: Tuple[str, str], tuning_vars: List[str], tuning_dict: Dict[str, np.ndarray], ordering: List[int] = None, x_mask: np.ndarray = None, y_mask: np.ndarray = None, **kwargs) -> matplotlib.figure.Figure:
    assert len(plot_vars)==2

    title = kwargs.get('title', None)

    margin = kwargs.get('margin', 0.05)
    key_prefix = kwargs.get('key_prefix', '')
    limit_n_neurons = kwargs.get('limit_n_neurons', None)
    label = kwargs.get('label', 'Activity')

    x_var, y_var = plot_vars
    x_bins = tuning_vars[f'{key_prefix}{x_var}']['bins']
    y_bins = tuning_vars[f'{key_prefix}{y_var}']['bins']

    try:
        tuning_grid = tuning_dict[f'{key_prefix}{x_var}_to_{y_var}_tuning'].copy()
        tuning_bins_size = tuning_dict[f'{key_prefix}{x_var}_to_{y_var}_tuning_bin_size'].copy()
    except KeyError:
        tuning_grid = tuning_dict[f'{key_prefix}{y_var}_to_{x_var}_tuning'].copy().transpose((0, 2, 1))
        tuning_bins_size = tuning_dict[f'{key_prefix}{y_var}_to_{x_var}_tuning_bin_size'].copy().transpose((0, 2, 1))

    if x_mask is not None:
        x_bins = x_bins[x_mask]
        tuning_grid = tuning_grid[:,x_mask,:]
    if y_mask is not None:
        y_bins = y_bins[y_mask]
        tuning_grid = tuning_grid[:,:,y_mask]

    if ordering is None:
        ordering = [i for i in range(tuning_grid.shape[0])]

    if limit_n_neurons is not None:
        ordering = ordering[:limit_n_neurons]
        tuning_grid = tuning_grid[:limit_n_neurons]
    
    clt_mask = tuning_bins_size < 5
    tuning_grid[clt_mask] = np.nan


    if key_prefix == 'z-':
        norm = matplotlib.colors.CenteredNorm(vcenter=0, halfrange=2, clip=True)
        cmap = 'seismic'
    else:
        tuning_grid += 1e-10
        norm = matplotlib.colors.LogNorm(vmin=1e-10, vmax=np.max(tuning_grid))
        cmap = 'turbo'
    
    def _plot_joint_tuning(ax, neuron):
        neuron_i = ordering.index(neuron)
        ax.imshow(tuning_grid[neuron_i].T, cmap=cmap, label='Activity', aspect='auto', extent=[x_bins[0], x_bins[-1], y_bins[0], y_bins[-1]], norm=norm)

    def _make_legend(fig, ax):
        im_artist = ax[0,0].images[0]
        cbar_ax = fig.add_axes([0.75, margin/2, 1 - 0.75 - margin, margin/4])
        cbar = fig.colorbar(im_artist, cax=cbar_ax, orientation='horizontal')
        cbar.set_label(label)

    # Use neuron_by_neuron_plot template to create plot
    fig = neuron_by_neuron_plot(n_neurons=tuning_grid.shape[0],
                                 plot_closure=_plot_joint_tuning,
                                 x_label=tuning_vars[f'{key_prefix}{x_var}']['title'],
                                 y_label=tuning_vars[f'{key_prefix}{y_var}']['title'], 
                                 legend_closure = _make_legend, 
                                 ordering=ordering, **kwargs)
    
    if title is not None:
        fig.suptitle(title, fontsize=20)

    return fig

In [ ]:
fig = bivar_tuning_plot(('hd', 'av'), tuning_vars, tuning_dict, key_prefix='z-', limit_n_neurons=None, label='Z-Scored Firing Rate', title='Head Direction vs Angular Velocity Tuning - JAL005 Baseline')

In [ ]:
fig = bivar_tuning_plot(('sd', 'av'), tuning_vars, tuning_dict, key_prefix='z-', limit_n_neurons=None, label='Z-Scored Firing Rate per Frame', title='Head-Shelter Angle vs Angular Velocity Tuning - JAL005 Baseline')

# Dimensionality

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
import numpy as np
from math import ceil, sqrt

pca_objs = []
pca_activities = []
for exp_idx in behave_df.unique('exp_idx').sort('exp_idx').select('exp_idx').to_numpy().flatten():
    exp_behave_df = behave_df.filter(pl.col('exp_idx')==exp_idx)
    exp_spikes_df = spikes_df.filter(pl.col('exp_idx')==exp_idx)

    activity = exp_spikes_df.select(['i', 't', 'r']).sort(by=['i', 't']).pivot(index='i', on='t', values='r').drop('i').to_numpy()

    exp_pca = PCA(n_components=10)
    exp_pca_activity = exp_pca.fit_transform(activity.T).T

    pca_objs.append(exp_pca)
    pca_activities.append(exp_pca_activity)





In [ ]:

fig = plt.figure(figsize=(12, 10))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 3])

# Top half: bar plot of explained variance ratio
ax_top = fig.add_subplot(gs[0, 0])
n_exps = len(pca_objs)
x_vals = np.arange(10)
bar_width = 0.8 / n_exps

for i, pca in enumerate(pca_objs):
    ax_top.bar(x_vals + i*bar_width, np.cumsum(pca.explained_variance_ratio_[:10]), 
                width=bar_width, label=f'exp_idx {i}')
ax_top.set_xlabel('Principal Component')
ax_top.set_ylabel('Explained Variance Ratio')
ax_top.legend()

# Bottom half: 3D PCA plots for each experiment
sub_gs = gs[1, 0].subgridspec(
    nrows=int(ceil(n_exps / ceil(sqrt(n_exps)))),
    ncols=int(ceil(sqrt(n_exps)))
)

for i, pc_data in enumerate(pca_activities):
    row = i // sub_gs.ncols
    col = i % sub_gs.ncols
    ax_3d = fig.add_subplot(sub_gs[row, col], projection='3d')
    ax_3d.scatter(pc_data[0], pc_data[1], pc_data[2], s=1)
    ax_3d.set_title(f'exp_idx {i}')

plt.tight_layout()
plt.show()

# Other Plots

## Firing Rate Distribution

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(15, 10))

ax[0].hist(spikes_df.select("z").to_numpy(), bins=50)
ax[0].set_title("Z-scores distribution")
ax[0].set_yscale("log")

ax[1].hist(spikes_df.select("r").to_numpy(), bins=50)
ax[1].set_title("Firing rates distribution")
ax[1].set_yscale("log")

example_i = 0
plot_time = 10
example_cluster = spikes_df.unique('i').select("i")[example_i]
example_cluster_df = spikes_df.filter(pl.col("i")==example_cluster['i']).sort("t")
t = example_cluster_df.select("t").to_numpy() * frame_time / 60
r = example_cluster_df.select("r").to_numpy()
c = example_cluster_df.select("c").to_numpy()
z = example_cluster_df.select("z").to_numpy()
plot_frames = int(plot_time // frame_time)


ax[2].plot(t[:plot_frames], r[:plot_frames], label="Firing rate (100ms window)", color='red', zorder=100)
ax[2].plot(t[:plot_frames], c[:plot_frames], label="Number of spikes", color='green', zorder=10)
ax[2].plot(t[:plot_frames], z[:plot_frames], label="Z-scored Firing Rate", color='blue', zorder=0)

plt.show()

## Position Distribution

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors


xbins = np.linspace(0, 1024, 26)
ybins = np.linspace(0, 1024, 26)
hdir_bins = np.linspace(-np.pi, np.pi, 101)

filtered_df = (
    behave_df
    .filter(pl.col("speed") < 1)
    .with_columns([
        pl.col("mouse_x_position").map_elements(lambda v: np.digitize(v, xbins) - 1, return_dtype=pl.Int64).alias("x_bin"),
        pl.col("mouse_y_position").map_elements(lambda v: np.digitize(v, ybins) - 1, return_dtype=pl.Int64).alias("y_bin"),
        pl.col("hdir").map_elements(lambda v: np.digitize(v, hdir_bins) - 1, return_dtype=pl.Int64).alias("h_bin"),
    ])
)

coverage = (
    filtered_df
    .group_by(["exp_idx", "x_bin", "y_bin"])
    .agg(pl.col("h_bin").n_unique().alias("n_hdir_bins"))
    .filter(pl.col("n_hdir_bins") > len(hdir_bins) * 0.5)
)

filtered_df = filtered_df.join(
    coverage,
    on=["exp_idx","x_bin","y_bin"],
    how="inner"
)


fig = plt.figure(figsize=(15,10))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.05, 0.05], wspace=1)

ax = fig.add_subplot(gs[0, 0])

heatmap_full, xedges, yedges = np.histogram2d(
    behave_df['mouse_x_position'].to_numpy(),
    behave_df['mouse_y_position'].to_numpy(),
    bins=[xbins, ybins]
)

im = ax.imshow(
    heatmap_full.T,
    origin='lower',
    extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
    aspect='auto',
    cmap='hot'
)
ax.set_aspect('equal')

cax1 = fig.add_subplot(gs[0, 1])
cbar = fig.colorbar(im, cax=cax1)
cbar.set_label("Number of frames")

n_unique_h = coverage.select("n_hdir_bins").to_series().to_numpy()
vmin, vmax = n_unique_h.min(), n_unique_h.max()
norm = colors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

for (exp_idx_val, x_idx, y_idx, n_hdir) in coverage.iter_rows():
    color_val = cmap(norm(n_hdir))
    rect = plt.Rectangle(
        (xbins[x_idx], ybins[y_idx]),
        xbins[x_idx + 1] - xbins[x_idx],
        ybins[y_idx + 1] - ybins[y_idx],
        fill=False,
        edgecolor=color_val,
        linewidth=1.5
    )
    ax.add_patch(rect)

cax2 = fig.add_subplot(gs[0, 2])
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, cax=cax2)
cbar.set_label("Number of unique head directions")


fig.suptitle("2D Heatmap of Mouse Positions")

plt.show()
